# Planning optimalisatie — v1

Deze notebook is de "runner": hij roept de code in `src/` aan (data inladen, model bouwen, oplossen) en toont de resultaten. De logica zelf staat in `.py`-bestanden zodat die later ook door een dashboard herbruikt kan worden — hier zie je alleen de uitkomst.

Draai de cellen gewoon van boven naar onder (Shift+Enter).

In [ ]:
import sys
sys.path.insert(0, "src")  # zodat we data.py, model.py, solve.py kunnen importeren

from data import load_data
from model import build_model
from solve import (
    solve_lexicographic,
    extract_schedule,
    compare_headcount,
    compare_department_count_per_medewerker,
    compare_travel_pair,
)
from output import write_results

## 1. Data inladen

Leest `data/planning_input.xlsx` (nu nog de toy-data uit `src/make_toy_data.py`). Zodra je je eigen medewerkers/afdelingen/locaties in dat Excel-bestand zet, verandert er hier niets — alleen de uitkomst.

De inzetbare tijd per medewerker (kolom `Inzetbare tijd` in Employees) wordt in Excel zelf berekend: contracturen min de afwezigheids-/takenkolommen ernaast (opleiding, verlof, coördinerende taken, etc.). Python leest alleen die uitkomst.

In [ ]:
p = load_data("data/planning_input.xlsx")

print("Medewerkers:", p.E)
print("Afdelingen:", p.D)
print("Locaties:", p.L)
print("Te vermijden reispaar:", p.l1, "<->", p.l2)
print("Inzetbare tijd per medewerker:", p.avail)

## 2. Model bouwen

Variabelen ($x, y, y^{loc}, z, w$) en constraints 1 t/m 10 uit de wiskundige formulering. Constraints 8-10 zijn optioneel en periode-specifiek (headcount-cap, afdelingsgroep-cohesie, minimumuren) — ze doen niets tenzij je de bijbehorende velden in het Excel-bestand invult.

In [ ]:
m, v = build_model(p)  # PuLP-model, wordt opgelost met de open-source solver HiGHS

print("Aantal variabelen:", m.numVariables())
print("Aantal constraints:", m.numConstraints())

## 3. Oplossen (lexicografisch)

Drie sequentiële solves: eerst minimale afwijking, dan binnen dat optimum minimale reistijd, dan binnen dat optimum minimaal aantal medewerkers per afdeling.

In [ ]:
# msg=True toont de HiGHS-solver-log voor elke van de 3 stappen
result = solve_lexicographic(m, p, v, msg=False)

print("Aantal gewijzigde toewijzingen:", result["deviation"])
print("Aantal medewerkers op beide reis-locaties:", result["travel"])
print("Aantal (medewerker, afdeling)-koppels:", result["headcount"])

## 4. De resulterende planning

In [ ]:
schedule = extract_schedule(p, v)
schedule = schedule[["locatie", "afdeling", "medewerker", "uren"]]
schedule = schedule.sort_values(["locatie", "afdeling", "medewerker"]).reset_index(drop=True)
schedule

## 5. Planning per medewerker

Welke locaties en afdelingen elke medewerker krijgt toegewezen.

In [ ]:
per_medewerker = (
    schedule.groupby("medewerker")
    .agg(
        locaties=("locatie", lambda s: sorted(set(s))),
        afdelingen=("afdeling", lambda s: sorted(set(s))),
    )
    .reset_index()
    .sort_values("medewerker")
    .reset_index(drop=True)
)
per_medewerker

## 6. Verbeteringen t.o.v. de huidige planning

Alleen de afdelingen/medewerkers waar daadwerkelijk iets verandert (verschil = 0 wordt niet getoond), plus het reispaar ONO-LH.

In [ ]:
afdeling_verschil_alle = compare_headcount(p, schedule)
afdeling_verschil = afdeling_verschil_alle[afdeling_verschil_alle["verschil"] != 0]
display(afdeling_verschil)

medewerker_verschil_alle = compare_department_count_per_medewerker(p, schedule)
medewerker_verschil = medewerker_verschil_alle[medewerker_verschil_alle["verschil"] != 0]
display(medewerker_verschil)

travel_pair = compare_travel_pair(p, schedule)
display(travel_pair)

## 7. Exporteren naar Excel

Schrijft het rooster, de planning per medewerker en de verbeteringen-tabellen naar één Excel-bestand met een tabblad per tabel, zodat je dit kan delen zonder Python.

In [ ]:
output_path = "output/planning_resultaat.xlsx"
write_results(output_path, schedule, per_medewerker, afdeling_verschil, medewerker_verschil, travel_pair)
print(f"Resultaten geschreven naar {output_path}")